# WellbeingIQ · Notebook 5: GenAI Wellness Coach (Gemini)
**Modules:** `wellbeing_coach.py` (coach), `wellbeing_explain.py` (SHAP factors) · **App:** `app.py` (Streamlit)

> **Data note:** the model is trained on **synthetic** data. The coach gives general habit tips, never medical advice.

## Design in one picture
`student habits` → **model** (predicts category) → **SHAP** (top risk drivers and protective factors) → **Gemini** (rewords those facts into a short, kind message) → **safety checks** → message shown. If anything fails, **built-in guidance** is shown instead.

**Key design rules (from the proposal: uncertainty wording, fallback, responsible GenAI)**
1. The AI **never predicts**. It only rewords facts from the model and SHAP.
2. Only habit values and factor names are sent (no names, no IDs, no free text), so there is nothing a user can inject into the prompt.
3. The AI must answer in a fixed JSON shape, within length limits, and its text is **checked**: wording about diagnosis, medical conditions, medication, therapy or percentages is rejected.
4. Any problem (no key, wrong model name, timeout, bad JSON, unsafe wording) falls back to built-in tips, so the app never breaks or shows unsafe text.
5. Unreliable predictions (confidence below the Notebook 4 threshold) are marked "rough estimate".

In [1]:
import pandas as pd, numpy as np, json, time, warnings, os
from wellbeing_explain import WellbeingExplainer
from wellbeing_coach import WellnessCoach, SYSTEM_RULES, MODELS
warnings.filterwarnings('ignore')
ex = WellbeingExplainer(); X = pd.read_csv('X_test.csv')
print('models tried in order:', MODELS)

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


models tried in order: ['gemini-3.7-flash', 'gemini-2.5-flash']


## 1. What the AI is told

In [2]:
print(SYSTEM_RULES)

You are a supportive student wellness assistant inside a prototype app.
Rules you must follow:
- Use ONLY the facts given in the user message. Do not invent facts, numbers, statistics, percentages or probabilities.
- Give general habit suggestions only (sleep routine, screen time, activity, study planning, reaching out to people).
- Never diagnose, never mention medical or mental-health conditions, medication, therapy or treatment.
- Be warm, calm and non-judgemental. Never alarm or blame the student. Do not say the student 'is' at risk of a condition.
- If the prediction is marked uncertain, say plainly that this is a rough estimate.
- Make each tip concrete and tailored to the student's own numbers; choose the most useful ones from the changes listed.
- Never use the percent sign. If you mention an estimate, say 'points'.
- Reply with JSON only, in this exact shape:
  {"summary": "<2 to 3 short sentences>", "suggestions": ["<tip 1>", "<tip 2>", "<tip 3>"]}
- The summary must stay und

In [3]:
sent = []
probe = WellnessCoach(ex, generate_fn=lambda p,s,m: (sent.append(p), json.dumps({'summary':'x '*10,'suggestions':['a b c']}))[1])
probe.advise(X.iloc[5].to_dict()); print(sent[0])

Student facts (from the prediction model):
Predicted wellness category: Moderate
Prediction reliability: reasonably confident
Habits pushing toward lower wellbeing:
- Physical activity: 0 h (raises risk)
- Daily social media use: 4.9 h (raises risk)
- Phone unlocks per day: 143 (raises risk)
Habits helping wellbeing:
- Academic pressure: 4/10 (protective)
- Caffeine per day: 0 cups (protective)
- Notifications per day: 77 (protective)
Possible changes, with the model's estimated gain on the wellness meter (0 to 100):
- Sleep about 1 hour more (student's situation: You sleep 6.7 h a night): about 8 points higher on the wellness meter
- Trim late-night phone time by 30 min (student's situation: You use your phone 1.0 h late at night): about 6 points higher on the wellness meter
- Cut social media by 1 hour (student's situation: You spend 4.9 h a day on social media): about 5 points higher on the wellness meter
- Lighten the academic load (student's situation: Your pressure is 4 out of 10

## 2. Safety tests: what happens when the AI misbehaves (simulated Gemini responses)

In [4]:
good = json.dumps({'summary':'Your late-night phone use is the main thing pulling your wellbeing down, while good sleep is helping. This is a rough estimate.',
                   'suggestions':['Keep the last hour before bed screen-free.','Set a gentle daily limit for social media.','Keep up your regular sleep routine.']})
def boom(p,s,m): raise RuntimeError('404 model not found')
def slow(p,s,m): time.sleep(3); return good
cases = {'Valid JSON':lambda p,s,m:good, 'JSON inside code fence':lambda p,s,m:'```json\n'+good+'\n```',
         'Not JSON':lambda p,s,m:'Sure, here is some advice!', 'Mentions depression':lambda p,s,m:good.replace('rough estimate','sign of depression'),
         'Invents a percentage':lambda p,s,m:good.replace('main thing','main 40% thing'), 'API error (bad model name)':boom, 'Timeout':slow}
rows=[]
for name,fn in cases.items():
    r = WellnessCoach(ex, generate_fn=fn, timeout=1).advise(X.iloc[5].to_dict())
    rows.append({'Simulated Gemini behaviour':name,'Shown to the student':r['source'],'Reason AI text rejected':(r['ai_error'] or '')[:70]})
pd.DataFrame(rows)

,Simulated Gemini behaviour,Shown to the student,Reason AI text rejected
0,Valid JSON,Gemini AI,
1,JSON inside code fence,Gemini AI,
2,Not JSON,built-in guidance,gemini-2.5-flash: JSONDecodeError: Expecting v...
3,Mentions depression,built-in guidance,gemini-2.5-flash: ValueError: unsafe wording
4,Invents a percentage,built-in guidance,gemini-2.5-flash: ValueError: unsafe wording
5,API error (bad model name),built-in guidance,gemini-2.5-flash: RuntimeError: 404 model not ...
6,Timeout,built-in guidance,gemini-2.5-flash: timed out after 1s


Every failure ends in safe built-in guidance, and only a valid, checked answer is shown as AI-written.

## 3. Built-in guidance for three different students

In [5]:
coach = WellnessCoach(ex)
for i in [0,1,2]:
    r = coach.advise(X.iloc[i].to_dict())
    print(f"[{r['source']}] predicted {r['explanation']['predicted_category']} (uncertain={r['explanation']['uncertain']})\n{r['summary']}")
    for t in r['suggestions']: print('  -', t)
    print()

[built-in guidance] predicted Good (uncertain=False)
Your habits look healthy. Physical activity, sleep quality, sleep per night are working in your favour.
  - You sleep 8.0 h a night. Try going to bed 30 to 60 minutes earlier and keep the same wake-up time.
  - You spend 4.5 h a day on social media. Use app timers and start by trimming your most-used app.
  - You use your phone 1.3 h late at night. Charge it away from your bed and set a wind-down alarm.

[built-in guidance] predicted Moderate (uncertain=True)
Your habits look mixed. The biggest pulls toward lower wellbeing are academic pressure, late-night phone use, social support. The good news: trim late-night phone time by 30 min is estimated to help the most. This is only a rough estimate, so treat it as a starting point.
  - You use your phone 1.1 h late at night. Charge it away from your bed and set a wind-down alarm.
  - You sleep 8.1 h a night. Try going to bed 30 to 60 minutes earlier and keep the same wake-up time.
  - You

[built-in guidance] predicted Moderate (uncertain=True)
Your habits look mixed. The biggest pulls toward lower wellbeing are physical activity, social support, days offline per month. The good news: sleep about 1 hour more is estimated to help the most. This is only a rough estimate, so treat it as a starting point.
  - You sleep 7.8 h a night. Try going to bed 30 to 60 minutes earlier and keep the same wake-up time.
  - You use your phone 0.6 h late at night. Charge it away from your bed and set a wind-down alarm.
  - You spend 3.7 h a day on social media. Use app timers and start by trimming your most-used app.



## 4. Live Gemini call
Set your key first (never put it in code or in a public repository):
- Windows PowerShell: `$env:GEMINI_API_KEY="your-key"` · macOS/Linux: `export GEMINI_API_KEY="your-key"`

Model names change over time. If you see a "model not found" message, set `GEMINI_MODEL` to a model listed for your key (the coach also tries `gemini-2.5-flash` automatically).

> The live call could not be tested while this notebook was built (no key or internet access there). The cell below shows the real result on your machine.

In [6]:
live = WellnessCoach(ex)
if not live.ai_available:
    print('No GEMINI_API_KEY found. Set it and re-run this cell.')
else:
    r = live.advise(X.iloc[2].to_dict())
    print('Source:', r['source'], '| model:', r['model']); print(r['summary']); [print(' -', t) for t in r['suggestions']]
    if r['ai_error']: print('AI problem (fell back to built-in guidance):', r['ai_error'])

No GEMINI_API_KEY found. Set it and re-run this cell.


## 5. Responsible-AI notes
- **Privacy:** only habit values and factor names go to the API. No identifiers or free text.
- **Not medical:** the system prompt forbids diagnosis and medical wording, and the output filter enforces it. A fixed disclaimer is always shown.
- **Honesty about uncertainty:** low-confidence predictions are labelled a rough estimate.
- **Cost and availability:** results are cached per profile, and the app calls Gemini only when the button is pressed. If the API fails, built-in guidance appears.
- **Limits:** the AI can still word things imperfectly; the filter is rule-based and cannot catch everything. The tips are general and not personalised medical advice.